In [1]:
# 📓 LSTM Stock Forecasting Pipeline using FMP API (Daily Data)

# --- Step 1: Import Required Libraries ---
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, accuracy_score
import matplotlib.pyplot as plt
import requests


In [2]:
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Step 2: Fetch Daily Stock Data from FMP API ---
def fetch_fmp_daily_data(ticker, api_key, save_path='stock_data.csv'):
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}?serietype=line&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    if 'historical' not in data:
        raise ValueError("No historical data returned")
    df = pd.DataFrame(data['historical'])
    df['date'] = pd.to_datetime(df['date'])
    df = df.rename(columns={"date": "Date", "close": "Close_Prices"})
    df = df.sort_values(by='Date')
    df.to_csv(save_path, index=False)
    print(f"Saved {ticker} daily data to {save_path}")


Using device: cuda


In [3]:
# --- Step 3: Load and Process Data ---
def load_and_engineer_data(csv_path):
    df = pd.read_csv(csv_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by='Date')

    # Feature Engineering
    df['SMA_10'] = df['Close_Prices'].rolling(window=10).mean()
    df['EMA_10'] = df['Close_Prices'].ewm(span=10, adjust=False).mean()
    df['RSI'] = 100 - (100 / (1 + (df['Close_Prices'].pct_change().rolling(14).mean() /
                                   df['Close_Prices'].pct_change().rolling(14).std())))
    df['MACD'] = df['Close_Prices'].ewm(span=12, adjust=False).mean() - df['Close_Prices'].ewm(span=26, adjust=False).mean()
    df['Bollinger_High'] = df['Close_Prices'].rolling(20).mean() + (df['Close_Prices'].rolling(20).std() * 2)
    df['Bollinger_Low'] = df['Close_Prices'].rolling(20).mean() - (df['Close_Prices'].rolling(20).std() * 2)

    df['Target_Change'] = df['Close_Prices'].shift(-5) / df['Close_Prices'] - 1
    df['Target_Class'] = (df['Target_Change'] > 0).astype(int)

    df.dropna(inplace=True)
    return df


In [4]:
# --- Step 4: Create Sequences ---
def create_sequences(data, labels_reg, labels_cls, seq_length):
    X, y_reg, y_cls = [], [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y_reg.append(labels_reg[i+seq_length])
        y_cls.append(labels_cls[i+seq_length])
    return np.array(X), np.array(y_reg), np.array(y_cls)

# --- Step 5: Define LSTM Model

In [5]:

# --- Step 5: Define LSTM Model ---
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=3):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.reg_head = nn.Linear(hidden_size, 1)
        self.cls_head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.reg_head(last), torch.sigmoid(self.cls_head(last))


In [6]:
# --- Step 6: Training Function ---
def train_model(X_train, y_reg_train, y_cls_train, model, epochs=500):
    optimizer = optim.Adam(model.parameters(), lr=0.0005)
    loss_fn_reg = nn.MSELoss()
    loss_fn_cls = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        reg_pred, cls_pred = model(X_train)
        loss_reg = loss_fn_reg(reg_pred, y_reg_train)
        loss_cls = loss_fn_cls(cls_pred, y_cls_train)
        loss = loss_reg + loss_cls
        loss.backward()
        optimizer.step()
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}, Reg Loss: {loss_reg.item():.4f}, Cls Loss: {loss_cls.item():.4f}")


In [ ]:
# --- Step 7: Main ---
FMP_API_KEY = "JcmfjzcBwo5HGSiM5Yib7ylfG2PmSNzc"
ticker = "AAPL"
data_path = f"{ticker}_daily.csv"

fetch_fmp_daily_data(ticker, FMP_API_KEY, data_path)
df = load_and_engineer_data(data_path)

features = ['Close_Prices', 'SMA_10', 'EMA_10', 'RSI', 'MACD', 'Bollinger_High', 'Bollinger_Low']
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])

X_seq, y_reg, y_cls = create_sequences(X_scaled, df['Target_Change'].values, df['Target_Class'].values, 60)

# Train/test split (last year as test)
dates = df['Date'].values[60:]
split_date = np.datetime64('2023-01-01')
split_index = np.where(dates >= split_date)[0][0]

X_train, X_test = torch.tensor(X_seq[:split_index], dtype=torch.float32).to(device), torch.tensor(X_seq[split_index:], dtype=torch.float32).to(device)
y_reg_train = torch.tensor(y_reg[:split_index], dtype=torch.float32).unsqueeze(1).to(device)
y_cls_train = torch.tensor(y_cls[:split_index], dtype=torch.float32).unsqueeze(1).to(device)
y_reg_test = y_reg[split_index:]
y_cls_test = y_cls[split_index:]

model = LSTMModel(input_size=X_seq.shape[2]).to(device)
train_model(X_train, y_reg_train, y_cls_train, model, epochs=500)

# Evaluation
model.eval()
with torch.no_grad():
    reg_preds, cls_preds = model(X_test)
    reg_preds = reg_preds.cpu().numpy()
    cls_preds = (cls_preds.cpu().numpy() > 0.5).astype(int)

rmse = np.sqrt(mean_squared_error(y_reg_test, reg_preds))
accuracy = accuracy_score(y_cls_test, cls_preds)
print(f"\nTest RMSE: {rmse:.4f}, Directional Accuracy: {accuracy*100:.2f}%")

# Plot
plt.figure(figsize=(16, 5))
plt.plot(y_reg_test, label='Actual 5D Return')
plt.plot(reg_preds, label='Predicted 5D Return', linestyle='--')
plt.legend()
plt.title("LSTM Forecasted vs Actual Returns (5-Day)")
plt.grid(True)
plt.show()

Saved AAPL daily data to AAPL_daily.csv
Epoch 10, Reg Loss: 0.0040, Cls Loss: 0.6911
Epoch 20, Reg Loss: 0.0040, Cls Loss: 0.6911
Epoch 30, Reg Loss: 0.0039, Cls Loss: 0.6911
Epoch 40, Reg Loss: 0.0039, Cls Loss: 0.6910
Epoch 50, Reg Loss: 0.0039, Cls Loss: 0.6910
Epoch 60, Reg Loss: 0.0039, Cls Loss: 0.6908
Epoch 70, Reg Loss: 0.0039, Cls Loss: 0.6904
Epoch 80, Reg Loss: 0.0039, Cls Loss: 0.6909
Epoch 90, Reg Loss: 0.0039, Cls Loss: 0.6909
Epoch 100, Reg Loss: 0.0039, Cls Loss: 0.6908
Epoch 110, Reg Loss: 0.0039, Cls Loss: 0.6907
Epoch 120, Reg Loss: 0.0039, Cls Loss: 0.6905
Epoch 130, Reg Loss: 0.0039, Cls Loss: 0.6903
Epoch 140, Reg Loss: 0.0039, Cls Loss: 0.6900
Epoch 150, Reg Loss: 0.0039, Cls Loss: 0.6891
Epoch 160, Reg Loss: 0.0039, Cls Loss: 0.6886
Epoch 170, Reg Loss: 0.0039, Cls Loss: 0.6878
Epoch 180, Reg Loss: 0.0039, Cls Loss: 0.6874
Epoch 190, Reg Loss: 0.0039, Cls Loss: 0.6871
Epoch 200, Reg Loss: 0.0039, Cls Loss: 0.6872
Epoch 210, Reg Loss: 0.0039, Cls Loss: 0.6869
Epo